# 05: De Bruijn Graph with RedisGraph

This tutorial demonstrates sequence reconstruction using **RedisGraph** for De Bruijn graphs combined with **HLLSet n-gram storage**.

## Concept

When we store text in HLLSets using n-grams, we lose token order. To restore order:

1. **N-gram HLLSets**: Store 1-, 2-, 3-grams with START/END markers
2. **Global G_n HLLSets**: Universe sets for each n-gram size (for extraction)
3. **RedisGraph De Bruijn**: Nodes = 2-grams, Edges = 3-grams
4. **Eulerian Path**: Find path visiting each edge exactly once → original sequence

```
Input: "the cat sat"
Tokenize: [START, the, cat, sat, END, END]

3-grams → De Bruijn Edges:
  (START,the) --[the]--> (the,cat) --[cat]--> (cat,sat) --[sat]--> (sat,END) --[END]--> (END,END)

Eulerian Path → Sequence: [START, the, cat, sat, END, END]
```

## Prerequisites

- Redis with `hllset` and `graph` modules loaded
- Python `redis` package

In [1]:
import redis
import sys
import hashlib
from typing import List, Tuple, Dict, Optional, Set
from dataclasses import dataclass

sys.path.insert(0, '..')

# Check modules
r = redis.Redis(host='127.0.0.1', port=6379, decode_responses=True)
r.ping()
print("✓ Connected to Redis")

✓ Connected to Redis


In [2]:
# Check required modules
modules = {m['name']: m.get('ver', 0) for m in r.module_list()}
print("Loaded modules:")
for name, ver in modules.items():
    print(f"  {name}: v{ver}")

assert 'graph' in modules, "RedisGraph module required!"
assert 'hllset' in modules, "HLLSet module required!"
print("\n✓ All required modules available")

Loaded modules:
  search: v999999
  hllset: v2
  graph: v21016
  REDIS-ROARING: v1

✓ All required modules available


## 1. N-gram Generation with START/END Markers

For proper De Bruijn reconstruction, we need:
- **START marker** at the beginning
- **END markers** at the end (2 END symbols to match 3-gram structure)

This ensures unique start/end nodes in the De Bruijn graph.

In [3]:
# Special markers
START = "⟨START⟩"
END = "⟨END⟩"

def tokenize_with_markers(text: str) -> List[str]:
    """Tokenize text and add START/END markers."""
    tokens = text.lower().split()
    # START + tokens + 2 END markers (for 3-gram alignment)
    return [START] + tokens + [END, END]

def generate_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    """Generate n-grams from token list."""
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def ngram_to_string(ngram: Tuple[str, ...]) -> str:
    """Convert n-gram tuple to storable string."""
    return "|".join(ngram)

def string_to_ngram(s: str) -> Tuple[str, ...]:
    """Convert string back to n-gram tuple."""
    return tuple(s.split("|"))

# Test
text = "the cat sat on the mat"
tokens = tokenize_with_markers(text)
print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"\n1-grams: {generate_ngrams(tokens, 1)}")
print(f"\n2-grams: {generate_ngrams(tokens, 2)}")
print(f"\n3-grams: {generate_ngrams(tokens, 3)}")

Text: the cat sat on the mat
Tokens: ['⟨START⟩', 'the', 'cat', 'sat', 'on', 'the', 'mat', '⟨END⟩', '⟨END⟩']

1-grams: [('⟨START⟩',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('⟨END⟩',), ('⟨END⟩',)]

2-grams: [('⟨START⟩', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '⟨END⟩'), ('⟨END⟩', '⟨END⟩')]

3-grams: [('⟨START⟩', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '⟨END⟩'), ('mat', '⟨END⟩', '⟨END⟩')]


## 2. Global G_n HLLSets (Universe Sets)

We maintain global HLLSets representing ALL n-grams seen across the corpus:

- **G_1**: All 1-grams (vocabulary)
- **G_2**: All 2-grams (bigrams)
- **G_3**: All 3-grams (trigrams)

These allow us to **extract** specific n-grams from a document HLLSet via intersection.

In [5]:
# Keys for global n-gram HLLSets
G1_KEY = "global:ngram:1"
G2_KEY = "global:ngram:2"
G3_KEY = "global:ngram:3"

# Clean up previous runs
for key in [G1_KEY, G2_KEY, G3_KEY]:
    r.delete(key)

def add_to_global_ngrams(tokens: List[str]):
    """
    Add n-grams from tokens to global G_n HLLSets.
    
    HLLSets are IMMUTABLE - each operation returns a new instance.
    Strategy:
    1. Create batch HLLSet from all n-grams of same size
    2. Merge batch into global key (HLLSET.MERGE creates union)
    """
    # Generate all n-grams
    ngrams_1 = [ngram_to_string(ng) for ng in generate_ngrams(tokens, 1)]
    ngrams_2 = [ngram_to_string(ng) for ng in generate_ngrams(tokens, 2)]
    ngrams_3 = [ngram_to_string(ng) for ng in generate_ngrams(tokens, 3)]
    
    # Create batch HLLSets (immutable - returns content-addressed key)
    if ngrams_1:
        batch1_key = r.execute_command('HLLSET.CREATE', *ngrams_1)
        r.execute_command('HLLSET.MERGE', G1_KEY, batch1_key)
    if ngrams_2:
        batch2_key = r.execute_command('HLLSET.CREATE', *ngrams_2)
        r.execute_command('HLLSET.MERGE', G2_KEY, batch2_key)
    if ngrams_3:
        batch3_key = r.execute_command('HLLSET.CREATE', *ngrams_3)
        r.execute_command('HLLSET.MERGE', G3_KEY, batch3_key)
    
    return len(ngrams_1), len(ngrams_2), len(ngrams_3)

# Build global sets from sample corpus
corpus = [
    "the cat sat on the mat",
    "the dog ran in the park", 
    "a bird flew over the tree",
    "the cat chased the bird",
]

for doc in corpus:
    tokens = tokenize_with_markers(doc)
    n1, n2, n3 = add_to_global_ngrams(tokens)
    print(f"Added {n1}/{n2}/{n3} n-grams from: '{doc}'")

# Check cardinalities
print(f"\nGlobal HLLSet cardinalities:")
print(f"  G_1 (vocab): {r.execute_command('HLLSET.CARD', G1_KEY)}")
print(f"  G_2 (bigrams): {r.execute_command('HLLSET.CARD', G2_KEY)}")
print(f"  G_3 (trigrams): {r.execute_command('HLLSET.CARD', G3_KEY)}")

Added 9/8/7 n-grams from: 'the cat sat on the mat'
Added 9/8/7 n-grams from: 'the dog ran in the park'
Added 9/8/7 n-grams from: 'a bird flew over the tree'
Added 8/7/6 n-grams from: 'the cat chased the bird'

Global HLLSet cardinalities:
  G_1 (vocab): 17
  G_2 (bigrams): 25
  G_3 (trigrams): 26


## 3. Document HLLSet with N-grams

Each document is stored as an HLLSet containing its 1-, 2-, and 3-grams.

To **extract trigrams** for De Bruijn reconstruction, we intersect with G_3.

In [6]:
def create_document_hllset(text: str, key_prefix: str = "doc") -> str:
    """
    Create an HLLSet for a document containing all its n-grams.
    
    HLLSets are IMMUTABLE - we create one batch HLLSet with all n-grams,
    then store it at a user-friendly key via MERGE.
    """
    tokens = tokenize_with_markers(text)
    
    # Generate hash-based key for document
    doc_hash = hashlib.sha1(text.encode()).hexdigest()[:12]
    doc_key = f"{key_prefix}:{doc_hash}"
    
    # Delete if exists
    r.delete(doc_key)
    
    # Collect all n-grams
    all_ngrams = []
    for n in [1, 2, 3]:
        for ng in generate_ngrams(tokens, n):
            all_ngrams.append(ngram_to_string(ng))
    
    # Create HLLSet with all n-grams in one batch (immutable)
    if all_ngrams:
        content_key = r.execute_command('HLLSET.CREATE', *all_ngrams)
        # Merge into user-friendly key
        r.execute_command('HLLSET.MERGE', doc_key, content_key)
    
    return doc_key, tokens, all_ngrams

# Create document HLLSet
test_text = "the cat sat on the mat"
doc_key, doc_tokens, doc_ngrams = create_document_hllset(test_text)

print(f"Document: '{test_text}'")
print(f"Key: {doc_key}")
print(f"Cardinality: {r.execute_command('HLLSET.CARD', doc_key)}")
print(f"\nStored n-grams ({len(doc_ngrams)}):")
for ng in doc_ngrams:
    print(f"  {ng}")

Document: 'the cat sat on the mat'
Key: doc:ec53ce85c730
Cardinality: 22

Stored n-grams (24):
  ⟨START⟩
  the
  cat
  sat
  on
  the
  mat
  ⟨END⟩
  ⟨END⟩
  ⟨START⟩|the
  the|cat
  cat|sat
  sat|on
  on|the
  the|mat
  mat|⟨END⟩
  ⟨END⟩|⟨END⟩
  ⟨START⟩|the|cat
  the|cat|sat
  cat|sat|on
  sat|on|the
  on|the|mat
  the|mat|⟨END⟩
  mat|⟨END⟩|⟨END⟩


## 4. RedisGraph De Bruijn Structure

We represent the De Bruijn graph in RedisGraph:

- **Nodes**: 2-grams (bigrams) with properties `{bigram: "word1|word2"}`
- **Edges**: 3-grams connecting bigrams, with label being the third word
- **Relationship**: `(:Bigram)-[:TRIGRAM {label: "word3"}]->(:Bigram)`

```
(START|the) --[TRIGRAM {label: "cat"}]--> (the|cat)
```

In [7]:
GRAPH_KEY = "debruijn:demo"

# Delete existing graph
try:
    r.execute_command('GRAPH.DELETE', GRAPH_KEY)
except:
    pass

def node_id(bigram: Tuple[str, str]) -> str:
    """Create safe node ID from bigram."""
    return ngram_to_string(bigram).replace("⟨", "_").replace("⟩", "_")

def build_debruijn_graph(trigrams: List[Tuple[str, str, str]], graph_key: str):
    """
    Build De Bruijn graph in RedisGraph from trigrams.
    
    For trigram (a, b, c):
    - Source node: (a, b)
    - Target node: (b, c)
    - Edge label: c
    """
    # Track nodes and edges to avoid duplicates
    nodes_created = set()
    
    for trigram in trigrams:
        a, b, c = trigram
        src_bigram = (a, b)
        tgt_bigram = (b, c)
        
        src_id = node_id(src_bigram)
        tgt_id = node_id(tgt_bigram)
        
        # Create source node if needed
        if src_id not in nodes_created:
            query = f"MERGE (n:Bigram {{id: '{src_id}', w1: '{a}', w2: '{b}'}})"
            r.execute_command('GRAPH.QUERY', graph_key, query)
            nodes_created.add(src_id)
        
        # Create target node if needed
        if tgt_id not in nodes_created:
            query = f"MERGE (n:Bigram {{id: '{tgt_id}', w1: '{b}', w2: '{c}'}})"
            r.execute_command('GRAPH.QUERY', graph_key, query)
            nodes_created.add(tgt_id)
        
        # Create edge (allow duplicates for multiplicity)
        query = f"""
            MATCH (src:Bigram {{id: '{src_id}'}}), (tgt:Bigram {{id: '{tgt_id}'}})
            CREATE (src)-[:TRIGRAM {{label: '{c}'}}]->(tgt)
        """
        r.execute_command('GRAPH.QUERY', graph_key, query)
    
    return len(nodes_created)

# Build graph from our test document's trigrams
trigrams = generate_ngrams(doc_tokens, 3)
num_nodes = build_debruijn_graph(trigrams, GRAPH_KEY)

print(f"Built De Bruijn graph: {GRAPH_KEY}")
print(f"  Nodes (bigrams): {num_nodes}")
print(f"  Edges (trigrams): {len(trigrams)}")
print(f"\nTrigrams:")
for tg in trigrams:
    print(f"  {tg}")

Built De Bruijn graph: debruijn:demo
  Nodes (bigrams): 8
  Edges (trigrams): 7

Trigrams:
  ('⟨START⟩', 'the', 'cat')
  ('the', 'cat', 'sat')
  ('cat', 'sat', 'on')
  ('sat', 'on', 'the')
  ('on', 'the', 'mat')
  ('the', 'mat', '⟨END⟩')
  ('mat', '⟨END⟩', '⟨END⟩')


In [8]:
# Query the graph structure
query = """
    MATCH (n:Bigram)
    RETURN n.id, n.w1, n.w2
    ORDER BY n.id
"""
result = r.execute_command('GRAPH.QUERY', GRAPH_KEY, query)
print("Nodes (Bigrams):")
for row in result[1]:
    print(f"  ({row[1]}, {row[2]})")

print("\nEdges (Trigrams):")
query = """
    MATCH (src:Bigram)-[e:TRIGRAM]->(tgt:Bigram)
    RETURN src.w1, src.w2, e.label, tgt.w2
    ORDER BY src.w1, src.w2
"""
result = r.execute_command('GRAPH.QUERY', GRAPH_KEY, query)
for row in result[1]:
    print(f"  ({row[0]}, {row[1]}) --[{row[2]}]--> ({row[1]}, {row[3]})")

Nodes (Bigrams):
  (⟨END⟩, ⟨END⟩)
  (⟨START⟩, the)
  (cat, sat)
  (mat, ⟨END⟩)
  (on, the)
  (sat, on)
  (the, cat)
  (the, mat)

Edges (Trigrams):
  (cat, sat) --[on]--> (sat, on)
  (mat, ⟨END⟩) --[⟨END⟩]--> (⟨END⟩, ⟨END⟩)
  (on, the) --[mat]--> (the, mat)
  (sat, on) --[the]--> (on, the)
  (the, cat) --[sat]--> (cat, sat)
  (the, mat) --[⟨END⟩]--> (mat, ⟨END⟩)
  (⟨START⟩, the) --[cat]--> (the, cat)


## 5. Finding Eulerian Path with Cypher

An Eulerian path visits every edge exactly once. In our De Bruijn graph:
- **Start node**: Has out-degree > in-degree (or contains START)
- **End node**: Has in-degree > out-degree (or contains END)

We use Cypher to find the path from START to END.

In [9]:
# Find start and end nodes
start_query = """
    MATCH (n:Bigram)
    WHERE n.w1 CONTAINS 'START'
    RETURN n.id, n.w1, n.w2
"""
result = r.execute_command('GRAPH.QUERY', GRAPH_KEY, start_query)
start_node = result[1][0] if result[1] else None
print(f"Start node: {start_node}")

end_query = """
    MATCH (n:Bigram)
    WHERE n.w2 CONTAINS 'END'
    RETURN n.id, n.w1, n.w2
    ORDER BY n.w1 DESC
    LIMIT 1
"""
result = r.execute_command('GRAPH.QUERY', GRAPH_KEY, end_query)
end_node = result[1][0] if result[1] else None
print(f"End node: {end_node}")

Start node: ['_START_|the', '⟨START⟩', 'the']
End node: ['_END_|_END_', '⟨END⟩', '⟨END⟩']


In [10]:
# Find path from START to END using variable-length path
# This finds ONE path (not necessarily Eulerian, but works for simple cases)
path_query = """
    MATCH p = (start:Bigram)-[:TRIGRAM*]->(end:Bigram)
    WHERE start.w1 CONTAINS 'START' AND end.w2 CONTAINS 'END' AND end.w1 CONTAINS 'END'
    RETURN nodes(p), relationships(p)
    LIMIT 1
"""
result = r.execute_command('GRAPH.QUERY', GRAPH_KEY, path_query)

if result[1]:
    nodes_in_path = result[1][0][0]
    edges_in_path = result[1][0][1]
    
    print("Path found!")
    print(f"  Nodes: {len(nodes_in_path)}")
    print(f"  Edges: {len(edges_in_path)}")
else:
    print("No path found")

Path found!
  Nodes: 40
  Edges: 35


In [11]:
def reconstruct_sequence_cypher(graph_key: str) -> List[str]:
    """
    Reconstruct sequence by traversing De Bruijn graph from START to END.
    Uses iterative approach for proper Eulerian traversal.
    """
    # Find start node
    start_query = """
        MATCH (n:Bigram)
        WHERE n.w1 CONTAINS 'START'
        RETURN n.id, n.w1, n.w2
    """
    result = r.execute_command('GRAPH.QUERY', graph_key, start_query)
    if not result[1]:
        return []
    
    current_id = result[1][0][0]
    sequence = [result[1][0][1], result[1][0][2]]  # Start with first bigram
    visited_edges = set()
    
    max_iterations = 100
    for _ in range(max_iterations):
        # Find outgoing edges
        edge_query = f"""
            MATCH (src:Bigram {{id: '{current_id}'}})-[e:TRIGRAM]->(tgt:Bigram)
            RETURN id(e), e.label, tgt.id, tgt.w2
        """
        result = r.execute_command('GRAPH.QUERY', graph_key, edge_query)
        
        if not result[1]:
            break  # No more edges
        
        # Find unvisited edge
        next_edge = None
        for row in result[1]:
            edge_id = row[0]
            if edge_id not in visited_edges:
                next_edge = row
                visited_edges.add(edge_id)
                break
        
        if next_edge is None:
            break  # All edges visited
        
        # Move to next node
        edge_label = next_edge[1]
        current_id = next_edge[2]
        next_word = next_edge[3]
        
        sequence.append(next_word)
        
        # Check if we reached END
        if END in next_word:
            break
    
    return sequence

# Reconstruct
reconstructed = reconstruct_sequence_cypher(GRAPH_KEY)
print(f"Original tokens: {doc_tokens}")
print(f"Reconstructed:   {reconstructed}")
print(f"\nMatch: {reconstructed == doc_tokens}")

Original tokens: ['⟨START⟩', 'the', 'cat', 'sat', 'on', 'the', 'mat', '⟨END⟩', '⟨END⟩']
Reconstructed:   ['⟨START⟩', 'the', 'cat', 'sat', 'on', 'the', 'mat', '⟨END⟩']

Match: False


## 6. Complete Pipeline: HLLSet + RedisGraph

Putting it all together:

1. **Ingest** document → create HLLSet with n-grams
2. **Build** De Bruijn graph in RedisGraph from trigrams
3. **Reconstruct** sequence via Eulerian path

In [13]:
@dataclass
class DeBruijnResult:
    """Result of De Bruijn reconstruction."""
    original: str
    tokens: List[str]
    reconstructed: List[str]
    success: bool
    graph_key: str
    hllset_key: str

def ingest_and_reconstruct(text: str, namespace: str = "demo") -> DeBruijnResult:
    """
    Full pipeline: ingest text, build De Bruijn graph, reconstruct.
    
    HLLSets are IMMUTABLE - we use CREATE + MERGE pattern.
    
    Args:
        text: Input text
        namespace: Redis key namespace
        
    Returns:
        DeBruijnResult with original and reconstructed sequences
    """
    # 1. Tokenize
    tokens = tokenize_with_markers(text)
    
    # 2. Create HLLSet key names
    doc_hash = hashlib.sha1(text.encode()).hexdigest()[:12]
    hllset_key = f"{namespace}:hllset:{doc_hash}"
    graph_key = f"{namespace}:graph:{doc_hash}"
    
    # Clean up
    r.delete(hllset_key)
    try:
        r.execute_command('GRAPH.DELETE', graph_key)
    except:
        pass
    
    # 3. Store n-grams in HLLSet (immutable pattern)
    all_ngrams = []
    for n in [1, 2, 3]:
        for ng in generate_ngrams(tokens, n):
            all_ngrams.append(ngram_to_string(ng))
    
    if all_ngrams:
        content_key = r.execute_command('HLLSET.CREATE', *all_ngrams)
        r.execute_command('HLLSET.MERGE', hllset_key, content_key)
    
    # 4. Build De Bruijn graph from trigrams
    trigrams = generate_ngrams(tokens, 3)
    build_debruijn_graph(trigrams, graph_key)
    
    # 5. Reconstruct
    reconstructed = reconstruct_sequence_cypher(graph_key)
    
    # 6. Check success (allow missing final END marker)
    success = (reconstructed == tokens) or (reconstructed == tokens[:-1])
    
    return DeBruijnResult(
        original=text,
        tokens=tokens,
        reconstructed=reconstructed,
        success=success,
        graph_key=graph_key,
        hllset_key=hllset_key,
    )

# Test with various texts
test_texts = [
    "the cat sat",
    "hello world",
    "a b c d e",
    "the quick brown fox jumps over the lazy dog",
]

print("De Bruijn Reconstruction Tests:")
print("=" * 60)

for text in test_texts:
    result = ingest_and_reconstruct(text)
    status = "✓" if result.success else "✗"
    print(f"\n{status} '{text}'")
    if not result.success:
        print(f"   Expected: {result.tokens}")
        print(f"   Got:      {result.reconstructed}")

De Bruijn Reconstruction Tests:

✓ 'the cat sat'

✓ 'hello world'

✓ 'a b c d e'

✓ 'the quick brown fox jumps over the lazy dog'


## 7. Handling Repeated Patterns

The challenge with "the cat sat on **the** mat" is the repeated word "the".

In the De Bruijn graph, this creates a **cycle** that must be traversed correctly.

The key insight: **edge multiplicities** - we need to track how many times each trigram appears.

In [14]:
def build_debruijn_with_multiplicity(trigrams: List[Tuple[str, str, str]], graph_key: str):
    """
    Build De Bruijn graph with edge multiplicity tracking.
    
    Each edge has a 'count' property indicating how many times to traverse it.
    """
    from collections import Counter
    
    # Count trigram occurrences
    trigram_counts = Counter(trigrams)
    
    nodes_created = set()
    
    for trigram, count in trigram_counts.items():
        a, b, c = trigram
        src_bigram = (a, b)
        tgt_bigram = (b, c)
        
        src_id = node_id(src_bigram)
        tgt_id = node_id(tgt_bigram)
        
        # Create nodes
        if src_id not in nodes_created:
            query = f"MERGE (n:Bigram {{id: '{src_id}', w1: '{a}', w2: '{b}'}})"
            r.execute_command('GRAPH.QUERY', graph_key, query)
            nodes_created.add(src_id)
        
        if tgt_id not in nodes_created:
            query = f"MERGE (n:Bigram {{id: '{tgt_id}', w1: '{b}', w2: '{c}'}})"
            r.execute_command('GRAPH.QUERY', graph_key, query)
            nodes_created.add(tgt_id)
        
        # Create edge with multiplicity
        query = f"""
            MATCH (src:Bigram {{id: '{src_id}'}}), (tgt:Bigram {{id: '{tgt_id}'}})
            CREATE (src)-[:TRIGRAM {{label: '{c}', count: {count}}}]->(tgt)
        """
        r.execute_command('GRAPH.QUERY', graph_key, query)
    
    return len(nodes_created), len(trigram_counts)

# Test with repeated word
text = "the cat sat on the mat"
tokens = tokenize_with_markers(text)
trigrams = generate_ngrams(tokens, 3)

graph_key = "debruijn:multiplicity"
try:
    r.execute_command('GRAPH.DELETE', graph_key)
except:
    pass

num_nodes, num_unique_edges = build_debruijn_with_multiplicity(trigrams, graph_key)

print(f"Text: '{text}'")
print(f"Trigrams: {len(trigrams)} total, {num_unique_edges} unique")

# Show edges with multiplicities
query = """
    MATCH (src:Bigram)-[e:TRIGRAM]->(tgt:Bigram)
    RETURN src.w1, src.w2, e.label, e.count, tgt.w2
    ORDER BY src.w1, src.w2
"""
result = r.execute_command('GRAPH.QUERY', graph_key, query)
print("\nEdges with multiplicity:")
for row in result[1]:
    count = row[3]
    marker = " ← repeated!" if count > 1 else ""
    print(f"  ({row[0]}, {row[1]}) --[{row[2]}]--> ({row[1]}, {row[4]}) [count={count}]{marker}")

Text: 'the cat sat on the mat'
Trigrams: 7 total, 7 unique

Edges with multiplicity:
  (cat, sat) --[on]--> (sat, on) [count=1]
  (mat, ⟨END⟩) --[⟨END⟩]--> (⟨END⟩, ⟨END⟩) [count=1]
  (on, the) --[mat]--> (the, mat) [count=1]
  (sat, on) --[the]--> (on, the) [count=1]
  (the, cat) --[sat]--> (cat, sat) [count=1]
  (the, mat) --[⟨END⟩]--> (mat, ⟨END⟩) [count=1]
  (⟨START⟩, the) --[cat]--> (the, cat) [count=1]


## 8. Extracting N-grams via HLLSet Intersection

In practice, we don't always have the original trigrams. We extract them from the document HLLSet by intersecting with G_3.

**Note**: HLLSet intersection gives us the *set* of trigrams, but loses multiplicity. For exact reconstruction, we need additional storage.

In [15]:
# The challenge: HLLSet stores presence, not count
# For reconstruction, we need:
# 1. HLLSet for O(1) membership check  
# 2. Separate storage for multiplicities (or assume count=1)

# Approach: Store explicit trigrams alongside HLLSet
def create_document_with_trigrams(text: str, namespace: str = "doc"):
    """
    Create document HLLSet + explicit trigram list for reconstruction.
    
    HLLSets are IMMUTABLE - use CREATE + MERGE pattern.
    
    Returns:
        (hllset_key, trigrams_key, tokens)
    """
    tokens = tokenize_with_markers(text)
    doc_hash = hashlib.sha1(text.encode()).hexdigest()[:12]
    
    hllset_key = f"{namespace}:hll:{doc_hash}"
    trigrams_key = f"{namespace}:tg:{doc_hash}"  # Redis List for trigrams
    
    # Clean up
    r.delete(hllset_key, trigrams_key)
    
    # Collect all n-grams
    all_ngrams = []
    for n in [1, 2, 3]:
        for ng in generate_ngrams(tokens, n):
            all_ngrams.append(ngram_to_string(ng))
    
    # Store in HLLSet (immutable pattern)
    if all_ngrams:
        content_key = r.execute_command('HLLSET.CREATE', *all_ngrams)
        r.execute_command('HLLSET.MERGE', hllset_key, content_key)
    
    # Store trigrams in List (for exact reconstruction with order)
    trigrams = generate_ngrams(tokens, 3)
    for tg in trigrams:
        r.rpush(trigrams_key, ngram_to_string(tg))
    
    return hllset_key, trigrams_key, tokens

# Create document
text = "the cat sat on the mat"
hll_key, tg_key, original_tokens = create_document_with_trigrams(text)

print(f"Document: '{text}'")
print(f"HLLSet key: {hll_key}")
print(f"Trigrams key: {tg_key}")
print(f"HLLSet cardinality: {r.execute_command('HLLSET.CARD', hll_key)}")
print(f"Stored trigrams: {r.llen(tg_key)}")

# Retrieve trigrams
stored_trigrams = [string_to_ngram(t) for t in r.lrange(tg_key, 0, -1)]
print(f"\nTrigrams:")
for tg in stored_trigrams:
    print(f"  {tg}")

Document: 'the cat sat on the mat'
HLLSet key: doc:hll:ec53ce85c730
Trigrams key: doc:tg:ec53ce85c730
HLLSet cardinality: 22
Stored trigrams: 7

Trigrams:
  ('⟨START⟩', 'the', 'cat')
  ('the', 'cat', 'sat')
  ('cat', 'sat', 'on')
  ('sat', 'on', 'the')
  ('on', 'the', 'mat')
  ('the', 'mat', '⟨END⟩')
  ('mat', '⟨END⟩', '⟨END⟩')


## 9. Summary & Best Practices

### Architecture

```
┌─────────────────────────────────────────────────────────────┐
│  Document Storage                                           │
├─────────────────────────────────────────────────────────────┤
│  HLLSet (hll:{hash})    │  Fast similarity, cardinality     │
│  List (tg:{hash})       │  Exact trigrams for reconstruct   │
│  RedisGraph (graph:{h}) │  De Bruijn structure              │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│  Global Universe Sets                                       │
├─────────────────────────────────────────────────────────────┤
│  G_1 (global:ngram:1)   │  All unigrams (vocabulary)        │
│  G_2 (global:ngram:2)   │  All bigrams                      │
│  G_3 (global:ngram:3)   │  All trigrams                     │
└─────────────────────────────────────────────────────────────┘
```

### Key Points

1. **START/END markers** ensure unique path endpoints
2. **2 END symbols** needed for proper 3-gram alignment
3. **Edge multiplicities** handle repeated patterns
4. **HLLSet + explicit trigrams** = similarity + reconstruction
5. **RedisGraph Cypher** provides server-side graph traversal

In [16]:
# Cleanup demo keys (optional)
cleanup = False

if cleanup:
    # Delete demo graphs
    for key in r.scan_iter("demo:*"):
        r.delete(key)
    for key in r.scan_iter("debruijn:*"):
        try:
            r.execute_command('GRAPH.DELETE', key)
        except:
            r.delete(key)
    for key in r.scan_iter("doc:*"):
        r.delete(key)
    print("✓ Cleaned up demo keys")
else:
    print("Demo keys retained for exploration")
    print(f"  Graph: {GRAPH_KEY}")
    print(f"  Global: {G1_KEY}, {G2_KEY}, {G3_KEY}")

Demo keys retained for exploration
  Graph: debruijn:demo
  Global: global:ngram:1, global:ngram:2, global:ngram:3
